# RAG Pipeline - Vector DB & Retrieval
**Βήμα 1:** Text Embeddings με all-MiniLM-L6-v2 (HuggingFace API)  
**Βήμα 2:** Δημιουργία Vector DB (ChromaDB)  
**Βήμα 3:** Retrieval + Answer Generation με Llama-3.1-8B (Ollama)

## Cell 1: Εγκατάσταση βιβλιοθηκών

In [ ]:
!pip install chromadb sentence-transformers requests --quiet

## Cell 2: Imports & Config

In [ ]:
import sys
sys.path.insert(0, ".")

from rag_preprocessor import RAGPreprocessor
from rag_retriever import RAGRetriever
from config import MANUALS_FOLDER

print("Imports OK!")


## Cell 3: Συνάρτηση Embeddings μέσω HuggingFace API

In [ ]:
# Το embedding γίνεται πλέον εσωτερικά από RAGPreprocessor και RAGRetriever
# Για νέα PDFs που δεν έχουν γίνει indexed:
preprocessor = RAGPreprocessor()
results = preprocessor.process_all(MANUALS_FOLDER)
if results:
    for f, (t, v) in results.items():
        print(f"  {f}: {t} text + {v} vision chunks")
else:
    print("Όλα τα PDFs είναι ήδη indexed.")


## Cell 4: Φόρτωση δεδομένων (chunks + vision descriptions)

In [ ]:
retriever = RAGRetriever()
print(f"Collection: {retriever.count()} documents")
print(f"Sources: {retriever.list_sources()}")


## Cell 5: Δημιουργία Vector DB με ChromaDB

In [ ]:
# Το indexing γίνεται πλέον από RAGPreprocessor
# Το collection είναι ήδη έτοιμο μέσω RAGRetriever
print(f"Vector DB έτοιμο: {retriever.count()} documents")


## Cell 6: Συνάρτηση Retrieval

In [ ]:
# Test retrieval
test_results = retriever.retrieve("installation requirements")
print(f"Test retrieval: {len(test_results)} results")
for r in test_results:
    print(f"  [{r['metadata']['type']}] p.{r['metadata'].get('page','?')} "
          f"score={r['score']:.3f} | {r['text'][:80]}...")


## Cell 7: Συνάρτηση Answer Generation με Llama-3.1-8B

In [ ]:
# Test answer generation
answer, docs = retriever.query("What are the safety warnings?")
print("ANSWER:")
print(answer)


## Cell 8: Το πλήρες RAG Pipeline

In [ ]:
# Πλήρες RAG pipeline — ένα call
answer, docs = retriever.query(
    "What are the installation requirements for the machine?",
    n_results=5
)
print(answer)


In [ ]:
import chromadb
from pathlib import Path

client     = chromadb.PersistentClient(path=str(Path("data/vector_db")))
collection = client.get_collection("rag_manuals")

pdf_stem = "B-85444EN-1_03_02_Robodrill_Operator_Manual"  # π.χ. "TruLaser_2030_manual"

text_result = collection.get(
    where={"$and": [
        {"source": {"$eq": pdf_stem}},
        {"type":   {"$eq": "text"}}
    ]},
    include=[]
)
vision_result = collection.get(
    where={"$and": [
        {"source": {"$eq": pdf_stem}},
        {"type":   {"$eq": "image_description"}}
    ]},
    include=[]
)

print(f"Text chunks:   {len(text_result['ids'])}")
print(f"Vision chunks: {len(vision_result['ids'])}")


## Cell 9: Δοκιμή του RAG

In [ ]:
# Δοκίμασε με μια ερώτηση σχετική με το manual σου
answer = rag_query(
    query="What are the installation requirements for the machine?",
    n_results=5
)

In [ ]:
# Δοκίμασε κι άλλη ερώτηση
answer = rag_query(
    query="What are the safety warnings I should be aware of?",
    n_results=5
)